# Agent 365 - Registry Ingester (Fabric) — recommended default

> **Status: supported (app-only, unattended) — this is the recommended default path** for landing the
> Agent 365 registry into the Lakehouse. As of **PAX `purview-v1.11.11`** and the current
> Microsoft Graph docs, the *Agent 365 catalog* endpoint supports **application permissions** and
> **`/v1.0`**, so this notebook runs **headless on a schedule** with a service principal — no
> interactive sign-in. (It was previously a delegated-only, interactive PREVIEW.)

## What this does

Pulls the tenant's **Agent 365 agent catalogue** (declarative agents, plugins, etc. - including
their data-access **capabilities/permissions**: OneDrive/SharePoint read, Graph connector, code
interpreter, image generation, uploaded files) into the Lakehouse Delta table `dbo.agents_365`
(the table the dashboard reads), keyed on `Title ID` (the `T_`-prefixed titleId).

## Requirements & caveats (read before using)

| Item | Detail |
|---|---|
| **Auth** | **App-only / client credentials** (service principal). Runs unattended. No user sign-in. |
| **Permissions** | **Application** permissions `CopilotPackages.Read.All` **+** `Application.Read.All`, **admin-consented**. |
| **Agent 365 licence** | **Still required in the tenant.** This is a *SKU* check, separate from permissions - a missing licence returns **`403`** (`Customer must be licensed for Agent 365`). |
| **Endpoint version** | Uses **`/v1.0`** (GA). Automatically falls back to **`/beta`** if a tenant hasn't surfaced v1.0 yet (PAX itself still calls beta). |
| **Point-in-time only** | No history; deleted agents disappear; `Date created`/`Created by` need a separate Purview audit-log join (out of scope here). |

**Relationship to the export lander:** `Copilot_Agent365_Lander.ipynb` (admin-center **export CSV**)
is a **fallback** for tenants that can't grant the app-registration permissions this notebook needs,
or for one-off / evaluation runs. **Prefer this notebook** for scheduled production pipelines — you
get the live capability detail, no CSV upload step, and an automated refresh. The two notebooks are
**alternatives** — they write to the same `dbo.agents_365` table, so running both in the same pipeline
would just clobber each other.

## Setup

- An **app registration** (service principal) with the two **Application** permissions above,
  admin-consented, plus a **client secret** (store it in Key Vault / a Fabric secret, not in code)
  or a **certificate** / **managed identity**.
- Install MSAL only if you use the managed-identity path; the default client-secret path uses
  `requests` and needs nothing extra.

*Endpoint, 28-column schema and capability fields align with the PAX Agent 365 enrichment output so
the dashboard's `Agents 365` query reads them unchanged. Ref: Microsoft Graph
`copilot/admin/catalog/packages` (v1.0) and PAX `purview-v1.11.11`.*

## 1. Configuration & app-only sign-in

**App-only (client-credentials)** flow - runs unattended, so this notebook can be a scheduled Fabric
job. No device-code, no browser. The service principal's admin-consented **Application** permissions
(`CopilotPackages.Read.All` + `Application.Read.All`) are carried in the token via the `.default`
scope.

In [ ]:
# === CONFIG ===
TENANT_ID    = '<your-tenant-guid>'
CLIENT_ID    = '<app-reg-client-id>'      # Application perms: CopilotPackages.Read.All + Application.Read.All (admin-consented)
TARGET_TABLE = 'dbo.agents_365'   # canonical table the dashboard reads (matches the lander)
WRITE_MODE   = 'overwrite'
ALLOW_EMPTY_SNAPSHOT = False      # set True only for an intentional empty first install

# Client secret - DO NOT hardcode. Pull from Key Vault / a Fabric-managed secret at runtime, e.g.:
#   CLIENT_SECRET = notebookutils.credentials.getSecret('https://<your-vault>.vault.azure.net/', 'Agent365AppSecret')
CLIENT_SECRET = '<from-key-vault>'

# App-only token via client credentials (mirrors Copilot_Audit_Log_Direct_Ingester.ipynb).
import requests

def _get_graph_token() -> str:
    url  = f'https://login.microsoftonline.com/{TENANT_ID}/oauth2/v2.0/token'
    data = {
        'client_id':     CLIENT_ID,
        'client_secret': CLIENT_SECRET,
        'scope':         'https://graph.microsoft.com/.default',
        'grant_type':    'client_credentials',
    }
    r = requests.post(url, data=data)
    r.raise_for_status()
    return r.json()['access_token']

TOKEN = _get_graph_token()
print('app-only token acquired:', bool(TOKEN))


## 2. Call the v1.0 catalog endpoint

Uses **`/v1.0`** (GA, app-only) and **auto-falls back to `/beta`** if v1.0 isn't yet exposed in the
tenant. List -> page via `@odata.nextLink` -> fetch per-package detail (the `elementDetails`
capability fields only appear at the detail level).

- **`403`** = missing **Agent 365 licence** in the tenant (SKU check), *or* the app hasn't been
  admin-consented for `CopilotPackages.Read.All`.
- **`401`** = token/consent problem, not a code bug.

In [ ]:
import requests

API_VERSION = 'v1.0'   # GA + app-only; falls back to beta below if a tenant hasn't surfaced v1.0 yet


def _base(v):
    return f'https://graph.microsoft.com/{v}/copilot/admin/catalog/packages'


def _validate_catalog_page(data, page_number):
    if not isinstance(data, dict):
        raise ValueError(f'Agent 365 catalog page {page_number} did not return an object.')
    if 'value' not in data:
        raise ValueError(f"Agent 365 catalog page {page_number} is missing required 'value'.")
    value = data['value']
    if not isinstance(value, list):
        raise ValueError(f"Agent 365 catalog page {page_number} returned a non-list 'value'.")
    next_link = data.get('@odata.nextLink')
    if next_link is not None and (not isinstance(next_link, str) or not next_link.strip()):
        raise ValueError(f"Agent 365 catalog page {page_number} returned an invalid '@odata.nextLink'.")
    for item_number, item in enumerate(value, start=1):
        if not isinstance(item, dict):
            raise ValueError(f'Agent 365 catalog page {page_number} item {item_number} is not an object.')
    return value, next_link


BASE = _base(API_VERSION)
H = {'Authorization': f'Bearer {TOKEN}'}

probe = requests.get(f'{BASE}?$top=1', headers=H, timeout=60)
if probe.status_code == 404 and API_VERSION == 'v1.0':
    API_VERSION = 'beta'
    BASE = _base(API_VERSION)
    probe = requests.get(f'{BASE}?$top=1', headers=H, timeout=60)
if probe.status_code == 403:
    raise PermissionError('403: Agent 365 catalog not accessible. Requires an Agent 365 LICENCE in '
                          'the tenant (SKU check) AND admin-consented Application permission '
                          'CopilotPackages.Read.All.')
if probe.status_code == 401:
    raise PermissionError('401: token/consent problem - check the app has admin consent for '
                          'CopilotPackages.Read.All + Application.Read.All (Application permissions).')
probe.raise_for_status()
print(f'Using {API_VERSION} endpoint.')

packages, url, seen_urls = [], BASE, set()
for page_number in range(1, 501):
    if url in seen_urls:
        raise ValueError(f'Agent 365 catalog paging loop detected at page {page_number}.')
    seen_urls.add(url)
    r = requests.get(url, headers=H, timeout=60)
    r.raise_for_status()
    page, next_url = _validate_catalog_page(r.json(), page_number)
    packages.extend(page)
    url = next_url
    print('packages in catalog so far:', len(packages))
    if not url:
        break
else:
    raise ValueError('Agent 365 catalog exceeded the 500-page safety cap.')

print('packages in catalog:', len(packages))


## 3. Map to the 28-column schema → Delta

Column names match the PAX `ConvertTo-Agent365Row` output so the dashboard's `Agents 365` query can
read them. `Title ID` is the primary/merge key. `Date created` / `Created by` are left blank here
(they need a separate Purview audit-log join — out of scope for this preview).

In [ ]:
# === SHAPE CATALOG -> canonical agents_365 schema ==============================
# Field names below are verified against a live Agent 365 catalog response, not
# assumed. The list endpoint returns inventory metadata; the per-package detail
# endpoint adds usage metrics and elementDetails. Observable malformed responses
# fail loudly so a bad snapshot cannot replace a good table.
import json as _json
from pyspark.sql import functions as F


def _join(value):
    if isinstance(value, list):
        return ';'.join(
            _json.dumps(item, ensure_ascii=False) if isinstance(item, (dict, list)) else str(item)
            for item in value if item is not None
        )
    return '' if value is None else str(value)


def _access(value):
    if value is None:
        return ''
    if not isinstance(value, list):
        raise ValueError('Agent 365 access collections must be lists when present.')
    out = []
    for item in value:
        if isinstance(item, dict):
            out.append(item.get('displayName') or item.get('id') or _json.dumps(item, ensure_ascii=False))
        else:
            out.append(str(item))
    return ';'.join(out)


def _normalise_registry_key(value):
    return (value or '').strip().upper()


def _detail_title_id(detail):
    raw = detail.get('id') or detail.get('titleId') or detail.get('packageId')
    if raw in (None, ''):
        raise ValueError('Agent 365 detail is missing id/titleId/packageId.')
    raw = str(raw)
    return raw if raw.startswith(('T_', 'P_')) else f'T_{raw}'


def _elements(detail):
    raw_groups = detail.get('elementDetails')
    if raw_groups is None:
        return '', '', ''
    if not isinstance(raw_groups, list):
        raise ValueError('elementDetails must be a list when present.')
    types, bots, commands = [], [], []
    for group_number, group in enumerate(raw_groups, start=1):
        if not isinstance(group, dict):
            raise ValueError(f'elementDetails group {group_number} is not an object.')
        element_type = group.get('elementType')
        if element_type:
            types.append(str(element_type))
        elements = group.get('elements')
        if elements is None:
            elements = []
        if not isinstance(elements, list):
            raise ValueError(f'elementDetails group {group_number} has a non-list elements payload.')
        for element_number, element in enumerate(elements, start=1):
            if not isinstance(element, dict):
                raise ValueError(f'elementDetails group {group_number} element {element_number} is not an object.')
            raw_definition = element.get('definition')
            if raw_definition in (None, ''):
                continue
            if isinstance(raw_definition, str):
                try:
                    definition = _json.loads(raw_definition)
                except _json.JSONDecodeError as exc:
                    raise ValueError(
                        f'Invalid elementDetails JSON for group {group_number} element {element_number}.'
                    ) from exc
            else:
                definition = raw_definition
            if not isinstance(definition, dict):
                raise ValueError(f'elementDetails group {group_number} element {element_number} did not decode to an object.')
            if definition.get('botId'):
                bots.append(str(definition['botId']))
            for command_list in definition.get('commandLists') or []:
                if not isinstance(command_list, dict):
                    raise ValueError(f'commandLists entry in group {group_number} element {element_number} is not an object.')
                for command in command_list.get('commands') or []:
                    if isinstance(command, dict) and command.get('title'):
                        commands.append(str(command['title']))
            for command in definition.get('commands') or []:
                if isinstance(command, dict) and command.get('title'):
                    commands.append(str(command['title']))
    return (
        ';'.join(dict.fromkeys(types)),
        ';'.join(dict.fromkeys(bots)),
        ';'.join(dict.fromkeys(commands)),
    )


def _dedupe_registry_rows(rows):
    seen = {}
    deduped = []
    for row in rows:
        key = _normalise_registry_key(row.get('Title ID'))
        if not key:
            raise ValueError('Agent 365 row is missing Title ID; refusing to write an ambiguous snapshot.')
        comparable = {col: '' if value is None else str(value).strip() for col, value in row.items()}
        prior = seen.get(key)
        if prior is None:
            seen[key] = comparable
            deduped.append(row)
            continue
        if comparable != prior:
            raise ValueError(f'Conflicting Agent 365 rows detected for {key!r}; refusing to overwrite a good snapshot.')
    return deduped


def _table_exists(table_name):
    return bool(spark.catalog.tableExists(table_name))


def _guard_registry_snapshot(rows, table_name):
    exists = _table_exists(table_name)
    if not rows and exists:
        raise ValueError(f'Fetched 0 Agent 365 rows; refusing to replace existing {table_name}.')
    if not rows and not ALLOW_EMPTY_SNAPSHOT:
        raise ValueError(
            f'Fetched 0 Agent 365 rows and {table_name} does not exist yet. '
            'Set ALLOW_EMPTY_SNAPSHOT = True only for an intentional empty first install.'
        )


rows = []
for package in packages:
    package_id = package.get('id') or package.get('titleId') or package.get('packageId')
    if package_id in (None, ''):
        raise ValueError('Agent 365 catalog list item is missing id/titleId/packageId.')
    detail_response = requests.get(f'{BASE}/{package_id}', headers=H, timeout=60)
    detail_response.raise_for_status()
    detail = detail_response.json()
    if not isinstance(detail, dict):
        raise ValueError(f'Agent 365 detail for {package_id!r} did not return an object.')

    element_types, bot_ids, commands = _elements(detail)
    title_id = _detail_title_id(detail)
    rows.append({
        'Agent name':         detail.get('displayName') or '',
        'Title ID':           title_id,
        'Version':            detail.get('version') or '',
        'Entra Agent ID':     detail.get('agentIdentityId') or '',
        'Bot Id':             bot_ids,
        'App Id':             detail.get('appId') or '',
        'Asset Id':           detail.get('assetId') or '',
        'Publisher':          detail.get('publisher') or '',
        'Agent creator':      detail.get('publisher') or '',
        'Agent creator ID':   detail.get('ownerId') or '',
        'Agent type (A365)':  detail.get('type') or '',
        'Created in':         detail.get('platform') or '',
        'Date created':       detail.get('createdDateTime') or '',
        'Last updated':       detail.get('lastModifiedDateTime') or '',
        'Agent description':  detail.get('shortDescription') or detail.get('longDescription') or '',
        'Categories':         _join(detail.get('categories')),
        'Supported in':       _join(detail.get('supportedHosts')),
        'Availability':       detail.get('availableTo') or '',
        'Status':             detail.get('deployedTo') or '',
        'Is Blocked':         str(detail.get('isBlocked', '') or ''),
        'Users shared':       _access(detail.get('sharedWithUsersAndGroups')),
        'Groups shared':      _access(detail.get('allowedUsersAndGroups')),
        'Element types':      element_types or _join(detail.get('elementTypes')),
        'Custom actions':     commands,
        'Active Users':       str(detail.get('activeUsers', '') or ''),
        'Total sessions':     str(detail.get('totalSessions', '') or ''),
        'Exception rate':     str(detail.get('exceptionRate', '') or ''),
        'Run Time':           str(detail.get('totalRunTimeInHours', '') or ''),
        'Last Activity Date': detail.get('lastUsedDateTime') or '',
        'Sensitivity': '',
        'Can read OneDrive and Sharepoint items': '', 'OneDrive and Sharepoint items': '',
        'Can read OneDrive files': '', 'OneDrive files': '', 'OneDrive sites': '',
        'Can read Sharepoint sites and files': '', 'Sharepoint files': '', 'Sharepoint sites': '',
        'Can extend to Graph connector': '', 'Graph connector details': '',
        'Can generate images using user prompt': '', 'Can use code interpreter': '',
        'Contains uploaded files': '', 'Uploaded files': '',
        'Environment Id': '', 'Instructions': '', 'Deployment': '', 'Risks': '',
    })

rows = _dedupe_registry_rows(rows)
_guard_registry_snapshot(rows, TARGET_TABLE)
if not rows:
    print(f'Writing explicit empty first snapshot to {TARGET_TABLE} (ALLOW_EMPTY_SNAPSHOT=True).')

rows = [{key: ('' if value is None else str(value)) for key, value in row.items()} for row in rows]
CANONICAL = [
    'Agent name', 'Supported in', 'Date created', 'Agent creator', 'Publisher',
    'Agent type (A365)', 'Version', 'Availability', 'Agent creator ID',
    'Agent description', 'Created in', 'Last updated', 'Custom actions',
    'Title ID', 'Sensitivity',
    'Can read OneDrive and Sharepoint items', 'OneDrive and Sharepoint items',
    'Can read OneDrive files', 'OneDrive files', 'OneDrive sites',
    'Can read Sharepoint sites and files', 'Sharepoint files', 'Sharepoint sites',
    'Can extend to Graph connector', 'Graph connector details',
    'Can generate images using user prompt', 'Can use code interpreter',
    'Contains uploaded files', 'Uploaded files', 'Status',
    'Active Users', 'Total sessions', 'Exception rate', 'Last Activity Date',
    'Deployment', 'Run Time', 'Risks',
]
for row in rows:
    for column in CANONICAL:
        row.setdefault(column, '')

if rows:
    ordered = CANONICAL + [column for column in rows[0] if column not in CANONICAL]
    df = spark.createDataFrame(rows).select(*[F.col(f'`{column}`') for column in ordered])
else:
    df = spark.createDataFrame([], ','.join(f'`{column}` string' for column in CANONICAL))

filled = {column: sum(1 for row in rows if row.get(column)) for column in (rows[0] if rows else {})}
print(f'packages shaped : {len(rows)}')
print('populated columns:')
for column in CANONICAL:
    count = filled.get(column, 0)
    if count:
        print(f'   {column:42} {count}/{len(rows)}')
blank = [column for column in CANONICAL if not filled.get(column)]
if blank:
    print(f'blank ({len(blank)}): {", ".join(blank)}')
    print('   -> these may be unavailable on the chosen source; only the Admin Center')
    print('      CSV export can populate the observability-only fields.')

(df.write.mode(WRITE_MODE)
   .option('overwriteSchema', 'true')
   .option('delta.columnMapping.mode', 'name')
   .option('delta.minReaderVersion', '2')
   .option('delta.minWriterVersion', '5')
   .format('delta').saveAsTable(TARGET_TABLE))
print(f'wrote -> {TARGET_TABLE} ({WRITE_MODE}) | columns: {len(df.columns)}')
